# spectraPyle GUI 

This notebook generates a validated configuration file for spectraPyle and launch the code. 

## Steps
1. Fill parameters below or LOAD existing GUI
2. Click **Validate Config**
3. Save configuration file to GUI/JSON/YAML (optional)
3. Click **Run spectraPyle** to start stacking

#### Mandatory actions -> (*), otherwise: default values

#### Developer: Salvatore Quai (salvatore.quai@unibo.it)
#### Version: 0.0.1


If validation fails → fix highlighted fields.


In [1]:
import json
import os
from pathlib import Path
import ipywidgets as w
from ipyfilechooser import FileChooser
from IPython.display import display, HTML,clear_output
from pydantic import TypeAdapter,BaseModel

import sys

import contextlib
import time
import threading

import importlib
import spectraPyle

import spectraPyle as stsp

import spectraPyle.stacking.stacking as stack
import spectraPyle.runtime.runtime_adapter  
import spectraPyle.schema.schema
importlib.reload(spectraPyle.runtime.runtime_adapter)  
importlib.reload(spectraPyle.schema.schema)  
importlib.reload(stack)
from spectraPyle.runtime.runtime_adapter import build_config_from_widgets, export_config_to_json, export_config_to_yaml
from spectraPyle.schema.schema import StackingConfigResolver
from spectraPyle.schema.schema import StackingConfig

In [2]:
PACKAGE_ROOT = Path(stsp.__file__).parent.parent.parent
#path_to_config_dir = PACKAGE_ROOT / "configs" / "GUI"
#print(path_to_config_dir)
#print (PACKAGE_ROOT)
CURRENT_PATH = PACKAGE_ROOT

In [3]:
def section(title):
    return w.HTML(f"<h3 style='color:#2c3e50'>{title}</h3>")

style = {'description_width': 'initial'}
#style = {'description_width': '300px'}

In [4]:
module_dir = os.path.dirname(spectraPyle.__file__)
rules_path = os.path.join(module_dir, "instruments", "instruments_rules.json")

with open(rules_path) as f:
    RULES = json.load(f)

In [5]:
# instrument:
import functools

instrument_w = w.Dropdown(
    options=["euclid", "desi"],
    value="euclid",
    description="Instrument",
    style=style,
)

survey_w = w.Dropdown(description="Survey", style=style)
datarelease_w = w.Dropdown(description="Data release", style=style)

# ---- per-grism state ----
grism_checkboxes = {}  # grism_name → Checkbox
grism_io_widgets = {}  # grism_name → dict with dir/file widgets
grism_section_box = w.VBox()  # rebuilt on survey change


def _make_grism_io_block(grism_name):
    """Build the dir+file chooser UI block for one grism."""
    dir_w = w.Text(
        description=f"{grism_name.capitalize()} dir",
        value=get_current_path(),
        style=style,
        layout=w.Layout(width='80%')
    )
    dir_fc = FileChooser(path=get_current_path(),
                         title=f"Select {grism_name} spectra directory",
                         select_default=False)
    dir_fc.show_only_dirs = True

    file_w = w.Text(
        description=f"{grism_name.capitalize()} file",
        placeholder="COMBINED_spectra",
        style=style,
        layout=w.Layout(width='80%')
    )
    file_fc = FileChooser(path=get_current_path(),
                          title=f"Select {grism_name} combined FITS file",
                          select_default=False)
    file_fc.filter_pattern = "*.fits"
    file_fc.show_only_files = True

    dir_box  = w.VBox([dir_fc])
    file_box = w.VBox([file_fc], layout=w.Layout(display='none'))

    def _on_dir_selected(chooser):
        if chooser.selected:
            dir_w.value = str(Path(chooser.selected).resolve())
    def _on_file_selected(chooser):
        if chooser.selected:
            full = chooser.selected
            file_w.value = os.path.splitext(os.path.basename(full))[0]
            dir_w.value  = os.path.dirname(full)
    dir_fc.register_callback(_on_dir_selected)
    file_fc.register_callback(_on_file_selected)

    outer = w.VBox([
        w.HTML(f"<b style='color:#2c7fb8'>Grism: {grism_name.upper()}</b>"),
        dir_box, file_box
    ], layout=w.Layout(display='none',
                       border='1px solid #ccc',
                       padding='8px',
                       margin='4px 0'))

    return dict(dir_w=dir_w, file_w=file_w,
                dir_box=dir_box, file_box=file_box, outer=outer)


def rebuild_grism_ui(*_):
    """Rebuild checkbox + I/O block list when instrument/survey changes."""
    inst   = instrument_w.value
    survey = survey_w.value
    if not survey:
        return

    available = RULES[inst]['surveys'][survey]['grisms']

    grism_checkboxes.clear()
    grism_io_widgets.clear()

    rows = []
    for g in available:
        cb = w.Checkbox(value=(len(available) == 1),
                        description=g.capitalize(),
                        style={'description_width': 'initial'})
        io_blk = _make_grism_io_block(g)
        if len(available) == 1:
            io_blk['outer'].layout.display = ''

        def _toggle(change, _g=g):
            grism_io_widgets[_g]['outer'].layout.display = '' if change['new'] else 'none'
            _update_io_mode_for_grism(_g)

        cb.observe(_toggle, names='value')

        grism_checkboxes[g] = cb
        grism_io_widgets[g] = io_blk
        rows.append(w.VBox([cb, io_blk['outer']]))

    grism_section_box.children = rows


def _update_io_mode_for_grism(grism):
    """Show dir chooser or file chooser depending on spectra_mode."""
    if grism not in grism_io_widgets:
        return
    blk  = grism_io_widgets[grism]
    mode = spectra_format_w.value if 'spectra_format_w' in dir() else None
    if mode == MODE_INDIVIDUAL:
        blk['dir_box'].layout.display  = ''
        blk['file_box'].layout.display = 'none'
    elif mode == MODE_COMBINED:
        blk['dir_box'].layout.display  = 'none'
        blk['file_box'].layout.display = ''
    elif mode == MODE_METADATA:
        blk['dir_box'].layout.display  = 'none'
        blk['file_box'].layout.display = 'none'


def update_all_grism_io_mode(*_):
    """Called when spectra_mode changes — refresh all active grism blocks."""
    for g in grism_checkboxes:
        if grism_checkboxes[g].value:
            _update_io_mode_for_grism(g)


def update_instrument(*_):
    inst = instrument_w.value
    rules = RULES[inst]
    survey_w.options = list(rules['surveys'].keys())
    survey_w.value   = rules['defaults']['survey']
    datarelease_w.options = rules['surveys'][survey_w.value]['data_release']
    datarelease_w.value   = rules['defaults']['data_release']
    rebuild_grism_ui()


def update_survey(*_):
    inst   = instrument_w.value
    survey = survey_w.value
    survey_rules = RULES[inst]['surveys'][survey]
    datarelease_w.options = survey_rules['data_release']
    datarelease_w.value   = RULES[inst]['defaults']['data_release']
    rebuild_grism_ui()


instrument_w.observe(update_instrument, names='value')
survey_w.observe(update_survey, names='value')

update_instrument()


In [6]:
# --------------------------------------------------
# Catalogue (hidden backend + chooser UI)
# --------------------------------------------------

def get_current_path():
    return str(Path(CURRENT_PATH).resolve())

dirIn_w = w.Text()
fileIn_w = w.Text()
fileExt_w = w.Text()

VALID_EXT = {"fits", "csv", "npz"}

# --------------------------------------------------
# File chooser
# --------------------------------------------------

file_fc = FileChooser(
    path=get_current_path(),
    title="Select catalogue file (csv, fits, npz)",
    filter_pattern=['*.fits', '*.npz', '*.csv'],
    select_default=False
)
file_fc.show_only_files = True

# --------------------------------------------------
# Label
# --------------------------------------------------

catalogue_label = w.HTML()

# --------------------------------------------------
# Helper: set selection (like spectra)
# --------------------------------------------------

def set_catalogue_selection(path):

    if not path:
        return

    directory, filename = os.path.split(path)
    name, ext = os.path.splitext(filename)

    ext = ext.replace(".", "").lower()

    if ext not in VALID_EXT:
        return

    # backend values
    dirIn_w.value = directory
    fileIn_w.value = name
    fileExt_w.value = ext

    # update chooser location
    file_fc.path = directory

    # update label
    #catalogue_label.value = (
    #    f"<b>Current catalogue:</b> {directory}/{name}.{ext}"
    #)

    update_catalogue_label()

# --------------------------------------------------
# On selection
# --------------------------------------------------

def on_file_selected(chooser):
    global CURRENT_PATH

    if chooser.selected:

        set_catalogue_selection(chooser.selected)

        # update CURRENT_PATH
        CURRENT_PATH = Path(dirIn_w.value)

        # sync spectra (as before)
        sync_current_path_to_spectra(CURRENT_PATH)

        # update output dir
        update_output_dir(dirIn_w.value)

        # update column names
        update_catalogue_columns()

file_fc.register_callback(on_file_selected)

# --------------------------------------------------
# External sync (for GUI loading)
# --------------------------------------------------

def sync_catalogue_to_ui(directory, filename, ext):

    global CURRENT_PATH

    path = os.path.join(directory, f"{filename}.{ext}")

    set_catalogue_selection(path)

    CURRENT_PATH = Path(dirIn_w.value)

    sync_current_path_to_spectra(CURRENT_PATH)
    update_output_dir(dirIn_w.value)

    update_catalogue_columns()

def update_catalogue_label():
    if dirIn_w.value and fileIn_w.value and fileExt_w.value:
        catalogue_label.value = (
            f"<b>Current catalogue:</b> "
            f"{dirIn_w.value}/{fileIn_w.value}.{fileExt_w.value}"
        )
    else:
        catalogue_label.value = "<b>Current catalogue:</b> None"

update_catalogue_label()

In [7]:
def update_output_dir(input_dir):
    base_output = os.path.join(input_dir, "output")
    #final_output = get_incremental_dir(base_output)    
    #dirOut_w.value = final_output
    dirOut_w.value = base_output
    
dirOut_w = w.Text(
    description="Output directory:",
    style=style
)

override_output_w = w.Checkbox(
    value=False,
    description="Customize output directory"
)

def toggle_output_edit(change):
    dirOut_w.disabled = not override_output_w.value

override_output_w.observe(toggle_output_edit, names="value")

# default: disabled
dirOut_w.disabled = True


def sync_current_path_to_spectra(path):
    set_dir_selection(path)
    spectra_file_fc.path = str(Path(path).resolve())
    
## QQQ

In [8]:
## OUTPUT FILENAME:

output_name_mode_w = w.Checkbox(
    value=False,  # False = AUTO
    description="Use custom output filename",
    style=style
)


output_name_custom_w = w.Text(
    placeholder="e.g. stacked_Halpha_Euclid_Q1",
    description="Custom name", style=style
)

output_name_w = w.Text(
    value="AUTO",  # special keyword → tells config to use filename_builder
    layout=w.Layout(display="none")
)

output_name_box = w.VBox()

def update_output_name(change=None):

    if not output_name_mode_w.value:
        # AUTO mode
        output_name_box.children = []
        output_name_w.value = "AUTO"

    else:
        # CUSTOM mode
        output_name_box.children = [output_name_custom_w]
        output_name_w.value = output_name_custom_w.value.strip()

def update_custom_name(change):
    if output_name_mode_w.value:
        output_name_w.value = output_name_custom_w.value.strip()

output_name_custom_w.observe(update_custom_name, names="value")

output_name_mode_w.observe(update_output_name, names="value")
update_output_name()

In [9]:
MODE_INDIVIDUAL = "individual fits"
MODE_METADATA = "metadata path"
MODE_COMBINED = "combined fits"

## format
spectra_format_w = w.RadioButtons(
    options=[MODE_INDIVIDUAL, MODE_COMBINED, MODE_METADATA],
    value=MODE_INDIVIDUAL,
    layout={'width': 'max-content'},
    description="Spectra format", style=style
)

In [10]:

# --------------------------------------------------
# Spectra directory (hidden backend + chooser UI)
# --------------------------------------------------

spectra_dir_w = w.Text(
    description="Spectra Directory",
    value=get_current_path()
)

spectra_dir_fc = FileChooser(
    path=get_current_path(),
    title="Select spectra directory",
    select_default=False
)
spectra_dir_fc.show_only_dirs = True

def set_dir_selection(path):
    """Safely update FileChooser selection"""
    path_str = str(Path(path).resolve())

    spectra_dir_w.value = path_str
    spectra_dir_fc.path = path_str

    # Force selection (needed for correct state)
    spectra_dir_fc._selected_path = path_str
    spectra_dir_fc._apply_selection()

def on_spectra_dir_selected(chooser):
    if chooser.selected:
        set_dir_selection(chooser.selected)

        # Sync file chooser
        spectra_file_fc.path = chooser.selected

spectra_dir_fc.register_callback(on_spectra_dir_selected)

spectra_dir_box = w.VBox([spectra_dir_fc])

# --------------------------------------------------
# Spectra datafile (Combined spectra only)
# --------------------------------------------------

spectra_datafile_w = w.Text(
    description="Spectra filename (FITS)",
    style=style,
    placeholder="COMBINED_spectra",
)

spectra_file_fc = FileChooser(
    path=get_current_path(),
    title="Select FITS file",
    select_default=False,
)
spectra_file_fc.filter_pattern = "*.fits"
spectra_file_fc.show_only_files = True

def on_spectra_file_selected(chooser):
    if chooser.selected:

        full_path = chooser.selected
        filename = os.path.basename(full_path)

        if not filename.lower().endswith(".fits"):
            return

        # set filename (no extension)
        spectra_datafile_w.value = os.path.splitext(filename)[0]

        # derive directory automatically
        directory = os.path.dirname(full_path)
        set_dir_selection(directory)

        

spectra_file_fc.register_callback(on_spectra_file_selected)

spectra_datafile_box = w.VBox(
    [spectra_file_fc],
    layout=w.Layout(display="none")
)

##
spectra_dir_label = w.HTML()

# --------------------------------------------------
# Logic: update directory visibility
# --------------------------------------------------

def update_spectra_dir(change=None):

    mode = spectra_format_w.value

    if mode == MODE_INDIVIDUAL:

        # show directory chooser
        spectra_dir_box.layout.display = ""
        spectra_dir_label.layout.display = ""

        # hide file chooser
        spectra_datafile_box.layout.display = "none"

        # ensure directory is set
        if not spectra_dir_fc.selected:
            set_dir_selection(get_current_path())

    elif mode == MODE_COMBINED:

        # hide directory chooser (we derive it)
        spectra_dir_box.layout.display = "none"

        # show label (still useful)
        spectra_dir_label.layout.display = ""

        # show file chooser
        spectra_datafile_box.layout.display = ""

        # ensure file chooser is in correct path
        spectra_file_fc.path = spectra_dir_w.value or get_current_path()

    elif mode == MODE_METADATA:

        spectra_dir_box.layout.display = "none"
        spectra_dir_label.layout.display = "none"
        spectra_datafile_box.layout.display = "none"

    else:

        spectra_dir_box.layout.display = "none"
        spectra_dir_label.layout.display = "none"
        spectra_datafile_box.layout.display = "none"



def update_spectra_label(change=None):
    spectra_dir_label.value = f"<b>Current spectra directory:</b> {spectra_dir_w.value}"

spectra_dir_w.observe(update_spectra_label, names="value")

# initialize
#update_spectra_label()

# --------------------------------------------------
# Logic: update file chooser behavior
# --------------------------------------------------

def update_spectra_datafile(change=None):

    mode = spectra_format_w.value

    if mode == MODE_METADATA:
        spectra_datafile_w.value = "metadata"
    elif mode != MODE_COMBINED:
        spectra_datafile_w.value = ""
        
# --------------------------------------------------
# External sync hook (IMPORTANT)
# --------------------------------------------------

def sync_current_path_to_spectra(path):
    """Call this when CURRENT_PATH changes"""
    set_dir_selection(path)
    spectra_file_fc.path = str(Path(path).resolve())

# --------------------------------------------------
# Initial trigger
# --------------------------------------------------

set_dir_selection(get_current_path())
update_spectra_dir()
update_spectra_datafile()




In [11]:
# Physics:
## Cosmology:
cosmo_H0 = w.FloatText(
    value=70,
    description=r"H$_0$"
)

cosmo_Om0 = w.FloatText(
    value=0.3,
    description="$\Omega_0$"
)




In [12]:
# Physics
## Redhift

ztype_w = w.Dropdown(
    options=[
        ("Rest frame", "rest_frame"),
        ("Observed frame", "observed_frame"),
        ("Minimum z", "minimum_z"),
        ("Maximum z", "maximum_z"),
        ("Median z", "median_z"),
        ("Custom value", "custom")
    ],
    value="rest_frame",
    description="Redshift type",
    style=style
)

z_custom_w = w.FloatText(
    value=None,
    description="Custom z",
    style=style
)

z_custom_box = w.VBox()

def update_ztype(change=None):

    if ztype_w.value == "custom":
        z_custom_box.children = [z_custom_w]
    else:
        z_custom_box.children = []
        
ztype_w.observe(update_ztype, names="value")
#update_ztype()


In [13]:
# Physics:
## Normalization
norm_w = w.Dropdown(
    options=["no_normalization", "custom", "median", "interval", "integral", "template"],
    value="median",
    description="Normalization",
    style=style
)

conservation_w = w.Dropdown(
    options=["flux", "luminosity"],
    value="luminosity",
    description="Conservation",
    style=style,
    layout=w.Layout(display="none")
)

def update_normalization(change=None):

    if norm_w.value == "no_normalization":
        # Must choose conservation
        conservation_w.layout.display = ""
        conservation_w.disabled = False

    else:
        # Must be None
        conservation_w.layout.display = "none"

#norm_w.observe(update_normalization, names="value")
#update_normalization()



In [14]:
## Interval wavelength for interval normalization, and statistic to apply to the flux in the interval:

lambda_norm_rest_value = None
interval_norm_statistics_value = None
conservation_value = None

lambda_min_w = w.FloatText(
    value=None,
    description="λ min",
    style=style
)

lambda_max_w = w.FloatText(
    value=None,
    description="λ max",
    style=style
)

interval_stat_w = w.Dropdown(
    options=["median", "mean", "maximum", "minimum"],
    value="median",
    description="Statistic",
    style=style
)

regular_box = w.VBox([
    w.HTML("<b>Flux conservation mode</b>"),
    conservation_w
])

interval_box = w.VBox([
    w.HTML("<b>Normalization wavelength interval (rest-frame)</b>"),
    w.HBox([lambda_min_w, lambda_max_w]),
    interval_stat_w
])

norm_dynamic_box = w.VBox()

def update_norm_ui(change=None):

    global lambda_norm_rest_value
    global interval_norm_statistics_value
    global conservation_value

    mode = norm_w.value

    # ---------- NO Normalization ----------
    if mode == "no_normalization":

        norm_dynamic_box.children = [regular_box]

        conservation_value = conservation_w.value
        lambda_norm_rest_value = None
        interval_norm_statistics_value = None

    # ---------- INTERVAL ----------
    elif mode == "interval":

        norm_dynamic_box.children = [interval_box]

        conservation_value = None
        lambda_norm_rest_value = [
            float(lambda_min_w.value),
            float(lambda_max_w.value)
        ]
        interval_norm_statistics_value = interval_stat_w.value

    # ---------- OTHER MODES ----------
    else:

        norm_dynamic_box.children = []

        conservation_value = None
        lambda_norm_rest_value = None
        interval_norm_statistics_value = None


def update_regular_values(change=None):
    global conservation_value

    if norm_w.value == "no_normalization":
        conservation_value = conservation_w.value

def update_interval_values(change=None):

    global lambda_norm_rest_value
    global interval_norm_statistics_value

    if norm_w.value == "interval":
        lambda_norm_rest_value = [
            float(lambda_min_w.value),
            float(lambda_max_w.value)
        ]
        interval_norm_statistics_value = interval_stat_w.value

        
## observers:
#norm_w.observe(update_norm_ui, names="value")

conservation_w.observe(update_regular_values, names="value")
update_regular_values()

lambda_min_w.observe(update_interval_values, names="value")
update_interval_values()
lambda_max_w.observe(update_interval_values, names="value")
update_interval_values()
interval_stat_w.observe(update_interval_values, names="value")
update_interval_values()


In [15]:
## Sampling

resampling_w = w.Dropdown(
    options=[
        ("Linear λ sampling", "lambda"),
        ("Log λ sampling", "log_lambda"),
        ("Shifted λ sampling", "lambda_shifted"),
        ("No resampling (observed frame only)", None)
    ],
    value="lambda",
    description="Resampling",
    style=style
)

resampling_note = w.HTML(
    "<i>Note: None allowed only in observed_frame and requires identical wavelength grids.</i>"
)


pixel_type_w = w.RadioButtons(
    options=[
        ("Manual pixel size", "manual"),
        ("Instrumental resolution", "instrumental")
    ],
    value="manual",
    description="Pixel mode",
    style=style
)

instrumental_resolution_note = w.HTML(
    "<i> Note: 'Instrumental resolution': the pixel size will be calculated according to the instrumental resolution provided in the instrument file.</i>"
)

pixel_size_w = w.FloatText(
    value=6,
    description="Δλ [Å]",
    style=style
)

n_nyq_w = w.FloatText(
    value=5,
    description="Nyquist sampling N",
    style=style
)


pixel_size_mode_value = None       
pixel_size_value = None            
n_nyq_value = None                 

## containers:
pixel_manual_box = w.VBox([pixel_size_w])
pixel_instr_box = w.VBox([n_nyq_w])

pixel_dynamic_box = w.VBox()
resampling_dynamic_box = w.VBox()

def update_resampling(change=None):

    global pixel_size_mode_value
    global pixel_size_value
    global n_nyq_value

    # Enforce ztype rule
    if ztype_w.value != "observed_frame" and resampling_w.value is None:
        resampling_w.value = "lambda"
        return

    if resampling_w.value is None:

        resampling_dynamic_box.children = []

        pixel_size_mode_value = None
        pixel_size_value = None
        n_nyq_value = None

    else:

        resampling_dynamic_box.children = [pixel_type_w, pixel_dynamic_box]
        update_pixel_mode()


def update_pixel_mode(change=None):

    global pixel_size_mode_value
    global pixel_size_value
    global n_nyq_value

    if resampling_w.value is None:
        return

    if pixel_type_w.value == "manual":

        pixel_dynamic_box.children = [pixel_manual_box]

        pixel_size_mode_value = "manual"
        pixel_size_value = float(pixel_size_w.value)
        n_nyq_value = None

    else:

        pixel_dynamic_box.children = [pixel_instr_box]

        pixel_size_mode_value = "instrumental"
        pixel_size_value = None
        n_nyq_value = float(n_nyq_w.value)


def update_manual_value(change=None):

    global pixel_size_value

    if pixel_type_w.value == "manual":
        pixel_size_value = float(pixel_size_w.value)

def update_instr_value(change=None):

    global n_nyq_value

    if pixel_type_w.value == "instrumental":
        n_nyq_value = float(n_nyq_w.value)


        
## observers:
resampling_w.observe(update_resampling, names="value")
#ztype_w.observe(update_resampling, names="value")

pixel_type_w.observe(update_pixel_mode, names="value")

pixel_size_w.observe(update_manual_value, names="value")
n_nyq_w.observe(update_instr_value, names="value")

update_resampling()



In [16]:
## ------- REFINING WAVELENGTH EXTENT (optional) ------- ##
lambda_banner = w.HTML("""
<div style="border-left:6px solid #2c7fb8; background:#eef6fb; padding:10px;">
Optional wavelength restriction for the stacked spectrum.<br><br>

If enabled, the stacked spectrum will only include wavelengths between:<br>
<b>left_edge × (1+z_stacking)</b> and <b>right_edge × (1+z_stacking)</b><br><br>

Useful to focus on emission lines or specific spectral regions.
</div>
""")

lambda_enable_w = w.Checkbox(
    value=False,
    description="Limit wavelength range",
    style=style
)

left_edge_w = w.FloatText(
    value=6150,
    description="Left edge",
    style=style
)

right_edge_w = w.FloatText(
    value=6750,
    description="Right edge",
    style=style
)

lambda_box = w.VBox()

def update_lambda_box(change=None):

    if lambda_enable_w.value:
        lambda_box.children = [
            lambda_banner,
            left_edge_w,
            right_edge_w
        ]
    else:
        lambda_box.children = []

# observer:
lambda_enable_w.observe(update_lambda_box, names="value")
update_lambda_box()

## ------- EDGE CROP (optional) ------- ##

edges_banner = w.HTML("""
<div style="border-left:6px solid #f28e2b; background:#fff4e6; padding:10px;">
Optional cropping of spectrum edges.<br><br>

Removes the first N and last M pixels before stacking.<br><br>

Uses Python slicing rules.<br>
Example: first=10, last=-10 → keeps spectrum[10:-10]
</div>
""")

edges_enable_w = w.Checkbox(
    value=False,
    description="Crop spectrum edges",
    style=style
)

first_pixel_w = w.IntText(
    value=10,
    description="First pixel",
    style=style
)

last_pixel_w = w.IntText(
    value=-10,
    description="Last pixel",
    style=style
)

edges_box = w.VBox()



def update_edges_box(change=None):

    if edges_enable_w.value:
        edges_box.children = [
            edges_banner,
            first_pixel_w,
            last_pixel_w
        ]
    else:
        edges_box.children = []

# Observer:
edges_enable_w.observe(update_edges_box, names="value")
update_edges_box()

In [17]:
def reset_widget(widget):
    """Reset dropdown/text widget to empty state"""
    try:
        if isinstance(widget, w.Dropdown):
            if UI_EMPTY in widget.options:
                widget.value = UI_EMPTY
            else:
                widget.value = widget.options[0]
        else:
            widget.value = ""
    except Exception:
        pass

def reset_metadata_fields():
    for wdg in [metadata_path_w, metadata_file_w, metadata_indx_w]:
        reset_widget(wdg)

def reset_redshift_field():
    reset_widget(redshift_column_input_w)

def reset_gal_ext_field():
    reset_widget(gal_ext_name_input_w)

def reset_custom_norm_field():
    reset_widget(custom_norm_input_w)

In [18]:
# =========================================================
# Catalogue Column Mapping 
# =========================================================

redshift_box = w.VBox()
metadata_box = w.VBox()
gal_ext_box = w.VBox()
custom_norm_box = w.VBox()

# -------------------------
# UI placeholder
# -------------------------
UI_EMPTY = "-- select column --"
EMPTY_OPTION = ["-- select catalogue first --"]

# -------------------------
# Helper: read catalogue
# -------------------------
def get_catalogue_columns(path):

    if not path:
        return []

    path = Path(path)
    ext = path.suffix.lower()

    try:
        if ext == ".csv":
            import pandas as pd
            cols = list(pd.read_csv(path, nrows=0).columns)

        elif ext == ".fits":
            from astropy.io import fits
            with fits.open(path) as hdul:
                for hdu in hdul:
                    if hasattr(hdu, "columns"):
                        cols = list(hdu.columns.names)
                        break
                else:
                    return []

        elif ext == ".npz":
            import numpy as np
            with np.load(path) as data:
                cols = list(data.keys())

        else:
            return []

        return sorted(cols)

    except Exception as e:
        print(f"Failed reading catalogue: {e}")
        return []

# -------------------------
# Dropdown widgets
# -------------------------
def make_dropdown(description):
    return w.Dropdown(
        options=EMPTY_OPTION,
        value=EMPTY_OPTION[0],
        description=description,
        style=style
    )

ID_column_w = make_dropdown("ID column")
redshift_column_input_w = make_dropdown("Redshift column")
metadata_path_w = make_dropdown("Metadata path")
metadata_file_w = make_dropdown("Metadata file")
metadata_indx_w = make_dropdown("Metadata HDU")
gal_ext_name_input_w = make_dropdown("E(B-V) column")
custom_norm_input_w = make_dropdown("Custom norm. column")

# -------------------------
# Update columns
# -------------------------
def update_catalogue_columns():

    if not (dirIn_w.value and fileIn_w.value and fileExt_w.value):
        columns = EMPTY_OPTION
    else:
        full_path = os.path.join(
            dirIn_w.value,
            f"{fileIn_w.value}.{fileExt_w.value}"
        )
        cols = get_catalogue_columns(full_path)

        if cols:
            columns = [UI_EMPTY] + cols
        else:
            columns = ["-- no columns found --"]

    widgets = [
        ID_column_w,
        redshift_column_input_w,
        metadata_path_w,
        metadata_file_w,
        metadata_indx_w,
        gal_ext_name_input_w,
        custom_norm_input_w
    ]

    for widget in widgets:
        current = widget.value

        widget.options = columns

        if current in columns:
            widget.value = current
        else:
            widget.value = UI_EMPTY if UI_EMPTY in columns else columns[0]

# -------------------------
# Redshift UI Logic
# -------------------------
def update_redshift_column(change=None):
    if ztype_w.value != "observed_frame":
        redshift_box.children = [redshift_column_input_w]
    else:
        reset_redshift_field()
        redshift_box.children = []

# -------------------------
# Metadata UI Logic
# -------------------------
def update_metadata(change=None):
    if spectra_format_w.value == MODE_METADATA:
        metadata_box.children = [
            w.HTML("<b>Metadata column names mapping</b>"),
            metadata_path_w,
            metadata_file_w,
            metadata_indx_w
        ]
    else:
        reset_metadata_fields()
        metadata_box.children = []

# -------------------------
# Galactic Extinction UI
# -------------------------
gal_ext_corr_w = w.Checkbox(
    value=False,
    description="Galactic extinction correction"
)

gal_ext_banner = w.HTML(
    """
    <div style="
        border-left: 6px solid #2c7fb8;
        background-color: #eef6fb;
        padding: 12px;
        margin-top: 10px;
        border-radius: 6px;
    ">
    <b>Galactic extinction corrections</b><br><br>
    Uses <b>Gordon+23 extinction curve</b> via <i>dust_extinction</i>.<br><br>
    Please cite the G23 model if used:<br>
    <a href="https://dust-extinction.readthedocs.io/en/latest/dust_extinction/references.html"
       target="_blank">
       dust_extinction reference page
    </a>
    </div>
    """,
    layout=w.Layout(display="none")
)

def update_gal_ext(change=None):
    if gal_ext_corr_w.value:
        gal_ext_box.children = [
            w.HTML("<i>Make sure E(B-V) column exists in catalogue</i>"),
            w.HTML("<i></i>"),
            gal_ext_name_input_w
        ]
        gal_ext_banner.layout.display = "block"
    else:
        reset_gal_ext_field()
        gal_ext_box.children = []
        gal_ext_banner.layout.display = "none"

# -------------------------
# Custom Norm UI
# -------------------------
def update_custom_norm(change=None):
    if norm_w.value == "custom":
        custom_norm_box.children = [
            w.HTML("<i>Make sure custom column exists in catalogue</i>"),
            custom_norm_input_w
        ]
    else:
        reset_custom_norm_field()
        custom_norm_box.children = []

# -------------------------
# Observers
# -------------------------
ztype_w.observe(update_redshift_column, names="value")
spectra_format_w.observe(update_metadata, names="value")
gal_ext_corr_w.observe(update_gal_ext, names="value")
norm_w.observe(update_custom_norm, names="value")

# -------------------------
# Initial Sync
# -------------------------
update_redshift_column()
update_metadata()
update_gal_ext()
update_custom_norm()

In [19]:
INSTRUMENT_EXTRA_QC = {
    "euclid": {
        "bad_pixels": True,
        "dithers": True,
    },
    "desi": {
        "bad_pixels": False,
        "dithers": False,
    },
    # future instruments:
    # "roman": {"bad_pixels": True, "dithers": False}
}

sigma_w = w.FloatSlider(
    value=4.,
    min=0.,
    max=5.,
    step=0.25,
    description="Sigma Clip", style=style
)


# bad pixels
pixel_mask_select_w = w.SelectMultiple(
    options=list(range(7)),     # bits 0–7
    value=(0, 6),               # default Euclid recommendation
    rows=7,
    description="Bad pixel bits", style=style
)

bad_pixel_note = w.HTML(
    "<i> Note: Multiple values can be selected with shift and/or ctrl (or command) pressed and mouse clicks or arrow keys.</i>"
)

## dithers
dither_banner = w.HTML("""
<div style="border-left:6px solid #2c7fb8; background:#eef6fb; padding:10px;">
Minimum number of dithers in the coadded 1D spectrum.<br>
Higher is better.<br><br> Note: Euclid Q1 max dithers = 4 (recommended ≥ 2).
</div>
""")

dither_w = w.IntSlider(
    value=2,
    min=1,
    max=50,
    step=1,
    description="Min dithers", style=style
)

instrument_qc_box = w.VBox()

def update_instrument_qc(change=None):

    inst = instrument_w.value
    caps = INSTRUMENT_EXTRA_QC.get(inst, {})

    children = []

    # ---- BAD PIXELS ----
    if caps.get("bad_pixels", False):
        #children += [badpix_banner, bad_pixel_note, pixel_mask_select_w]
        children += [bad_pixel_note, pixel_mask_select_w]

    # ---- DITHERS ----
    if caps.get("dithers", False):
        children += [dither_banner, dither_w]

    instrument_qc_box.children = children

## observers:
instrument_w.observe(update_instrument_qc, names="value")
update_instrument_qc()



In [20]:
bootstrap_R = w.BoundedIntText(
    value=300,
    min=0,
    max=1000,
    description="Bootstrap R"
)


In [21]:
parallel_enabled = w.Checkbox(
    value=True,
    description="Enable Multiprocessing"
)

parallel_cpu_frac = w.FloatSlider(
    value=0.9,
    min=0.1,
    max=0.95,
    step=0.05,
    description="CPU Fraction"
)


In [22]:
plot_enabled = w.Checkbox(
    value=True,
    description="Plot stacked spectrum"
)

In [23]:
def update_all_spectra_format_w(change=None):
    update_spectra_dir()
    update_spectra_datafile()
    update_metadata()
    update_all_grism_io_mode()

def update_all_ztype_w(change=None):
    update_ztype()
    update_resampling()
    update_redshift_column()

def update_all_norm_w(change=None):
    update_normalization()
    update_norm_ui()
    update_custom_norm()

spectra_format_w.observe(update_all_spectra_format_w, names="value")
ztype_w.observe(update_all_ztype_w, names="value")
norm_w.observe(update_all_norm_w, names="value")


In [25]:
# --------------------------------------------------
# CONFIG PATH
# --------------------------------------------------
path_to_config_dir = PACKAGE_ROOT / "configs" / "GUI"

# --------------------------------------------------
# OUTPUT WIDGET
# --------------------------------------------------
validation_load_gui = w.Output()

# --------------------------------------------------
# FILE CHOOSER
# --------------------------------------------------
load_GUI_fc = FileChooser(
    path=str(path_to_config_dir),
    title="Select GUI file (.gui)",
    filter_pattern="*.gui",
    select_default=False,
)
load_GUI_fc.reset()

selected_gui_label = w.HTML()

def on_gui_file_selected(chooser):
    if chooser.selected:
        selected_gui_label.value = f"<b>Selected:</b> {Path(chooser.selected).name}"
        load_GUI_btn.disabled = False
    else:
        selected_gui_label.value = ""
        load_GUI_btn.disabled = True

load_GUI_fc.register_callback(on_gui_file_selected)

# --------------------------------------------------
# LOAD BUTTON
# --------------------------------------------------
load_GUI_btn = w.Button(
    description="Load GUI",
    button_style="primary",
    disabled=True
)

# --------------------------------------------------
# RESTORE FUNCTION
# --------------------------------------------------
def restore_widgets_from_gui_dict(_):

    validation_load_gui.clear_output()

    with validation_load_gui:

        full_path = load_GUI_fc.selected
        if not full_path:
            print("No file selected.")
            return

        full_path = Path(full_path)
        if not full_path.exists():
            print(f"File not found: {full_path}")
            return

        try:
            with open(full_path) as f:
                payload = json.load(f)

            cfg = payload.get("gui_state", payload)

        except Exception as e:
            print(f"Failed loading config: {e}")
            return

        # -------------------------
        # Safe setters
        # -------------------------
        def safe_set(widget, value):
            try:
                if value in ("", None):
                    if hasattr(widget, "options") and UI_EMPTY in widget.options:
                        widget.value = UI_EMPTY
                    else:
                        widget.value = widget.options[0]
                else:
                    widget.value = value
            except Exception:
                pass

        def safe_set_2(widget_tuple, value_tuple):
            try:
                if isinstance(widget_tuple, (list, tuple)) and isinstance(value_tuple, (list, tuple)):
                    if len(widget_tuple) == len(value_tuple):
                        for w, v in zip(widget_tuple, value_tuple):
                            w.value = v
                else:
                    widget_tuple.value = value_tuple
            except Exception:
                pass

        # ==================================================
        # 🔵 PHASE 1 — CONTROL VARIABLES (CRITICAL)
        # ==================================================

        # ---- Instrument ----
        safe_set(instrument_w, cfg.get("instrument_name", instrument_w.value))
        update_instrument()

        safe_set(survey_w, cfg.get("survey_name", survey_w.value))
        # ---- Grisms (checkboxes) ----
        saved_grisms = cfg.get('grisms', [])
        saved_grism_io = cfg.get('grism_io', {})
        for g, cb in grism_checkboxes.items():
            cb.value = g in saved_grisms
        for g, io_blk in grism_io_widgets.items():
            gcfg = saved_grism_io.get(g, {})
            io_blk['dir_w'].value  = gcfg.get('spectra_dir', '') or ''
            io_blk['file_w'].value = gcfg.get('spectra_datafile', '') or ''
        safe_set(datarelease_w, cfg.get("data_release", datarelease_w.value))

        # ---- IO MODE (🔥 THIS FIXES YOUR BUG) ----
        safe_set(spectra_format_w, cfg.get("spectra_mode", spectra_format_w.value))

        # ---- Physics control ----
        safe_set(ztype_w, cfg.get("z_type", ztype_w.value))
        safe_set(norm_w, cfg.get("spectra_normalization", norm_w.value))

        # ==================================================
        # 🟡 PHASE 2 — TRIGGER UI UPDATES
        # ==================================================

        update_all_spectra_format_w()
        update_all_ztype_w()
        update_all_norm_w()
        update_catalogue_columns()

        # ==================================================
        # 🟢 PHASE 3 — SET DEPENDENT VALUES
        # ==================================================

        # -------------------------
        # IO
        # -------------------------
        safe_set(spectra_dir_w, cfg.get("spectra_dir", ""))

        sync_catalogue_to_ui(
            cfg.get("input_dir", ""),
            cfg.get("filename_in", ""),
            cfg.get("filename_in_extention", "csv"),
        )

        safe_set(dirOut_w, cfg.get("output_dir", ""))

        # -------------------------
        # Output filename
        # -------------------------
        filename_out = cfg.get("filename_out", "AUTO")
        if filename_out == "AUTO":
            safe_set(output_name_mode_w, "default")
        else:
            safe_set(output_name_mode_w, "custom")
            safe_set(output_name_custom_w, filename_out)

        update_output_name()

        # -------------------------
        # Column mapping
        # -------------------------
        safe_set(ID_column_w, cfg.get("ID_column_name", ""))
        safe_set(redshift_column_input_w, cfg.get("redshift_column_name", ""))

        # 🔥 Metadata now restored AFTER mode is set
        safe_set(metadata_path_w, cfg.get("metadata_path_column_name", ""))
        safe_set(metadata_file_w, cfg.get("metadata_file_column_name", ""))
        safe_set(metadata_indx_w, cfg.get("metadata_indx_column_name", ""))

        safe_set(gal_ext_name_input_w, cfg.get("gal_ext_column_name", ""))
        safe_set(custom_norm_input_w, cfg.get("custom_column_name", ""))

        # -------------------------
        # Physics
        # -------------------------
        safe_set(cosmo_H0, cfg.get("cosmo_H0", cosmo_H0.value))
        safe_set(cosmo_Om0, cfg.get("cosmo_Om0", cosmo_Om0.value))

        safe_set(z_custom_w, cfg.get("z_value", z_custom_w.value))
        safe_set(conservation_w, cfg.get("conservation", conservation_w.value))

        safe_set(
            interval_norm_statistics_value,
            cfg.get("interval_norm_statistics", interval_norm_statistics_value)
        )

        # -------------------------
        # Lambda
        # -------------------------
        safe_set_2(
            (lambda_min_w, lambda_max_w),
            cfg.get("lambda_norm_rest", lambda_norm_rest_value)
        )

        # -------------------------
        # Resampling
        # -------------------------
        safe_set(resampling_w, cfg.get("pixel_resampling_type", resampling_w.value))
        safe_set(pixel_type_w, cfg.get("pixel_size_type", pixel_type_w.value))
        safe_set(pixel_size_w, cfg.get("pixel_resampling", pixel_size_w.value))
        safe_set(n_nyq_w, cfg.get("nyquist_sampling", n_nyq_w.value))

        # -------------------------
        # QC / Performance
        # -------------------------
        safe_set(sigma_w, cfg.get("sigma_clipping_conditions", sigma_w.value))
        safe_set(bootstrap_R, cfg.get("bootstrapping_R", bootstrap_R.value))

        safe_set(
            pixel_mask_select_w,
            tuple(cfg.get("pixel_mask", list(pixel_mask_select_w.value)))
        )

        safe_set(dither_w, cfg.get("n_min_dithers", dither_w.value))

        safe_set(parallel_enabled, cfg.get("multiprocessing", parallel_enabled.value))
        safe_set(parallel_cpu_frac, cfg.get("max_cpu_fraction", parallel_cpu_frac.value))

        # -------------------------
        # Wavelength edges
        # -------------------------
        lambda_edges = cfg.get("lambda_edges_rest", False)

        if lambda_edges:
            lambda_enable_w.value = True
            left_edge_w.value = lambda_edges[0]
            right_edge_w.value = lambda_edges[1]
        else:
            lambda_enable_w.value = False

        spectrum_edges = cfg.get("spectrum_edges", False)

        if spectrum_edges:
            edges_enable_w.value = True
            first_pixel_w.value = spectrum_edges[0]
            last_pixel_w.value = spectrum_edges[1]
        else:
            edges_enable_w.value = False

        # -------------------------
        # Plot
        # -------------------------
        safe_set(plot_enabled, cfg.get("plot_results", plot_enabled.value))

        # -------------------------
        # Final UI sync
        # -------------------------
        update_gal_ext()

        print("✅ GUI loaded successfully")

# --------------------------------------------------
# LINK BUTTON
# --------------------------------------------------
load_GUI_btn.on_click(restore_widgets_from_gui_dict)


In [26]:
## preview config
def preview_config(_):
    with preview_out:
        clear_output()
        cfg = build_user_config()
        print(cfg)

def clean_value(widget):
    if isinstance(widget, w.Dropdown):
        if widget.value in (None, UI_EMPTY, "-- select catalogue first --", "-- no columns found --"):
            return ""
    return widget.value
    
def build_user_config():
    """ BUILD CONFIG FROM WIDGETS """
    
    lambda_edges_rest_value,spectrum_edges_value = build_wavelength_config()     
    
    return dict(
        ## IO
        input_dir = dirIn_w.value,
        output_dir = dirOut_w.value,

        filename_in = fileIn_w.value,
        filename_in_extention = fileExt_w.value,
        
        filename_out = output_name_w.value,
        
        spectra_mode = spectra_format_w.value,
        spectra_datafile = spectra_datafile_w.value,

        ID_column_name = clean_value(ID_column_w),
        redshift_column_name = clean_value(redshift_column_input_w),
        metadata_path_column_name = clean_value(metadata_path_w),
        metadata_file_column_name = clean_value(metadata_file_w),
        metadata_indx_column_name = clean_value(metadata_indx_w),
        galactic_extinction = gal_ext_corr_w.value, 
        gal_ext_column_name = clean_value(gal_ext_name_input_w),
        custom_column_name = clean_value(custom_norm_input_w),

        ## INSTRUMENT
        instrument_name = instrument_w.value,
        survey_name = survey_w.value,
        grisms = [g for g, cb in grism_checkboxes.items() if cb.value],
        grism_io = {
            g: {
                'spectra_dir': grism_io_widgets[g]['dir_w'].value,
                'spectra_datafile': grism_io_widgets[g]['file_w'].value or None,
            }
            for g, cb in grism_checkboxes.items() if cb.value
        },
        data_release = datarelease_w.value,
        
        ## PHYSICS
        cosmo_H0 = cosmo_H0.value,
        cosmo_Om0 = cosmo_Om0.value,
        
        ## Redshift
        z_type = ztype_w.value,
        z_value = z_custom_w.value,
        
        spectra_normalization = norm_w.value,
        conservation=conservation_value,
        
        spectrum_edges=spectrum_edges_value,
        lambda_edges_rest=lambda_edges_rest_value,
        
        interval_norm_statistics=interval_norm_statistics_value,
        lambda_norm_rest=lambda_norm_rest_value,
        
        pixel_resampling_type = resampling_w.value,
        pixel_size_type = pixel_size_mode_value,
        pixel_resampling = pixel_size_value,
        nyquist_sampling = n_nyq_value,
        
        ## QUALITY and PERFORMANCE
        sigma_clipping_conditions = sigma_w.value,
        bootstrapping_R = bootstrap_R.value,
        
        pixel_mask = list(pixel_mask_select_w.value),
        n_min_dithers = dither_w.value,
        
        multiprocessing = parallel_enabled.value,
        max_cpu_fraction = parallel_cpu_frac.value,
        
        plot_results = plot_enabled.value,
    )

def build_wavelength_config():

    if lambda_enable_w.value:
        lambda_edges_rest = [
            float(left_edge_w.value),
            float(right_edge_w.value)
        ]
    else:
        lambda_edges_rest = None

    if edges_enable_w.value:
        spectrum_edges = [
            int(first_pixel_w.value),
            int(last_pixel_w.value)
        ]
    else:
        spectrum_edges = None

    return lambda_edges_rest, spectrum_edges


preview_out = w.Output()

preview_btn = w.Button(description="Preview Config",style=dict(button_color='orange'))
preview_btn.on_click(preview_config)




In [27]:
validated_cfg = None

validate_btn = w.Button(
    description="Validate Config (*)",
    button_style="primary",
)

validation_output = w.Output()

def on_validate(_):
    global validated_cfg  
    
    validation_output.clear_output()
    
    try:
        with validation_output:
            cfg = build_config_from_widgets(build_user_config)
            cfg = StackingConfigResolver.resolve(cfg)
            print (f"validated config: {cfg}")
        
        validated_cfg = cfg
        
        with validation_output:
            print("✅ Config valid! Good Job!")
            print(f"Config version: {cfg.config_version}")
            
    except Exception as e:
        validated_cfg = None  
        with validation_output:
            print("❌ Validation failed!")
            print(e)

validate_btn.on_click(on_validate)


In [28]:
def on_export_GUI(_):
    global validated_cfg
    
    export_GUI_output.clear_output()
    
    path_to_config_dir = PACKAGE_ROOT / "configs" / "GUI"
    #path_to_config_dir = Path(os.getcwd()).parent / "configs" / "GUI"
    path_to_config_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        if validated_cfg is None:
            raise ValueError("No validated config available")
        
        gui_cfg = build_user_config()
        
        filename = save_GUI_name_w.value.strip()
        if not filename:
            raise ValueError("Filename is empty")

        if not filename.endswith(".gui"):
            filename += ".gui"

        full_path = path_to_config_dir / filename

        export_payload = {
            "gui_state": gui_cfg,
            "validated_config": validated_cfg.model_dump(mode="json")
        }

        with open(full_path, "w") as f:
            json.dump(export_payload, f, indent=2)

        with export_GUI_output:
            print(f"✅ GUI exported correctly: {full_path}")

    except Exception as e:
        with export_GUI_output:
            print("❌ Export GUI failed! see Log...")
        print("Export failed:", e)


export_GUI_output = w.Output()

save_GUI_name_w = w.Text(
    description="GUI filename",
    style=style
)

save_GUI_btn = w.Button(
    description="Save config (GUI)",
    style=dict(button_color='antiquewhite')
)

save_GUI_btn.on_click(on_export_GUI)


In [29]:
## common output name

save_config_name_w = w.Text(
    description="Config filename",
    style=style
)

In [30]:
def on_export_JSON(_):
    global validated_cfg
    
    export_JSON_output.clear_output()
    
    path_to_config_dir = PACKAGE_ROOT / "configs" / "JSON"
    #path_to_config_dir = Path(os.getcwd()).parent / "configs" / "JSON"
    path_to_config_dir.mkdir(parents=True, exist_ok=True)

    try:
        if validated_cfg is None:
            raise ValueError("No validated config available")

        filename = save_config_name_w.value.strip()
        if not filename:
            raise ValueError("Filename is empty")

        if not filename.endswith(".json"):
            filename += ".json"

        full_path = path_to_config_dir / filename

        export_config_to_json(validated_cfg, full_path)

        with export_JSON_output:
            print(f"✅ Export to JSON done: {full_path}")

    except Exception as e:
        with export_JSON_output:
            print("❌ Export to JSON failed! see Log...")
        print("Export failed:", e)

export_JSON_output = w.Output()

save_JSON_btn = w.Button(
    description="Save config (JSON)",
    style=dict(button_color='lightgreen')
)

save_JSON_btn.on_click(on_export_JSON)


In [31]:
def on_export_YAML(_):
    global validated_cfg
    
    export_YAML_output.clear_output()
    
    path_to_config_dir = PACKAGE_ROOT / "configs" / "YAML"
    #path_to_config_dir = Path(os.getcwd()).parent / "configs" / "YAML"
    path_to_config_dir.mkdir(parents=True, exist_ok=True)

    try:
        if validated_cfg is None:
            raise ValueError("No validated config available")

        filename = save_config_name_w.value.strip()
        if not filename:
            raise ValueError("Filename is empty")

        if not filename.endswith(".yaml"):
            filename += ".yaml"

        full_path = path_to_config_dir / filename

        export_config_to_yaml(validated_cfg, full_path)

        with export_YAML_output:
            print(f"✅ Export to YAML done: {full_path}")

    except Exception as e:
        with export_YAML_output:
            print("❌ Export to YAML failed! see Log...")
        print("Export failed:", e)

export_YAML_output = w.Output()

save_YAML_btn = w.Button(
    description="Save config (YAML)",
    style=dict(button_color='lightgreen')
)

save_YAML_btn.on_click(on_export_YAML)


In [32]:
# Output panel
run_spec_output = w.Output()

# Progress widgets
progress_bar = w.IntProgress(
    value=0,
    min=0,
    max=100,
    description='Running:',
    bar_style='',  # '', 'success', 'danger'
    style={'bar_color': '#2196F3'},
    orientation='horizontal'
)

progress_label = w.HTML(value="Starting...")
final_report = w.HTML(value="")

def get_unique_log_path(base_path: Path, suffix: str = ".log") -> Path:
    """Generate unique filename: spectraPyle_run.log → spectraPyle_run_1.log → spectraPyle_run_2.log"""
    
    # Create parent directories if missing
    base_path.parent.mkdir(parents=True, exist_ok=True)
    
    if not base_path.exists():
        return base_path
    
    stem = base_path.stem  # "spectraPyle_run"
    counter = 1
    while True:
        new_path = base_path.parent / f"{stem}_{counter}{suffix}"
        if not new_path.exists():
            return new_path
        counter += 1

import traceback

log_path_w = w.Text(value="", layout=w.Layout(display="none"))

#DEBUG_MODE = True  
DEBUG_MODE = False

def run_spectraPyle(_):
    global validated_cfg
    global stack

    run_spec_output.clear_output()
    final_report.value = ""
    progress_bar.value = 0
    progress_bar.bar_style = ''
    progress_label.value = "Initializing..."

    if validated_cfg is None:
        with run_spec_output:
            print("❌ No validated config available")
        return

    with run_spec_output:
        display(progress_bar)
        display(progress_label)
        display(final_report)

    start_time = time.time()
    stop_flag = {"done": False}

    # --------------------------------------------------
    # Animated progress
    # --------------------------------------------------
    def animate_progress():
        while not stop_flag["done"]:
            progress_bar.value = (progress_bar.value + 5) % 100
            elapsed = int(time.time() - start_time)
            progress_label.value = f"Running... {elapsed}s elapsed"
            time.sleep(0.5)

    thread = threading.Thread(target=animate_progress)
    thread.start()

    # --------------------------------------------------
    # Run pipeline
    # --------------------------------------------------

    try:
        path_to_log_file = get_unique_log_path(
            Path(f"{dirOut_w.value}/spectraPyle_run.log")
        )

        log_path_w.value = str(path_to_log_file)

        with run_spec_output:
            print(f"📝 Logging to: {path_to_log_file}")

        with open(path_to_log_file, "w") as logfile:

            # custom writer to log AND optionally print
            class Tee:
                def __init__(self, file, debug=False):
                    self.file = file
                    self.debug = debug
                    self.original_stdout = sys.__stdout__  # 👈 IMPORTANT
            
                def write(self, msg):
                    self.file.write(msg)
            
                    if self.debug:
                        self.original_stdout.write(msg)  # ✅ no recursion
            
                def flush(self):
                    self.file.flush()
                    if self.debug:
                        self.original_stdout.flush()

            tee = Tee(logfile, debug=DEBUG_MODE)

            with contextlib.redirect_stdout(tee), \
                 contextlib.redirect_stderr(tee):

                print("=== spectraPyle run started ===")
                print(f"Timestamp: {time.ctime()}")
                print(f"Config: {validated_cfg}")

                importlib.reload(stack)
                stack_run = stack.main(validated_cfg)

                print("=== spectraPyle run completed ===")

        stop_flag["done"] = True
        thread.join()

        progress_bar.value = 100
        progress_bar.bar_style = 'success'
        elapsed = round(time.time() - start_time, 2)

        final_report.value = f"""
        <div style="border:1px solid #4CAF50;padding:10px;border-radius:6px;background-color:#E8F5E9;">
            <b>✅ spectraPyle finished successfully</b><br>
            Total runtime: {elapsed} seconds<br>
            Log file: <code>{path_to_log_file}</code>
        </div>
        """

    except Exception as e:
        stop_flag["done"] = True
        thread.join()

        progress_bar.bar_style = 'danger'
        elapsed = round(time.time() - start_time, 2)

        # FULL traceback
        tb = traceback.format_exc()

        # Write traceback to log if possible
        try:
            with open(path_to_log_file, "a") as logfile:
                logfile.write("\n\n=== ERROR TRACEBACK ===\n")
                logfile.write(tb)
        except Exception:
            pass

        # Short message for UI
        short_error = str(e)

        final_report.value = f"""
        <div style="border:1px solid #F44336;padding:10px;border-radius:6px;background-color:#FFEBEE;">
            <b>❌ spectraPyle failed</b><br>
            Runtime before failure: {elapsed} seconds<br>
            Error: {short_error}<br>
            Log file: <code>{path_to_log_file}</code>
        </div>
        """

        # Optional: show full traceback in UI (only in debug)
        if DEBUG_MODE:
            with run_spec_output:
                print("\n🔍 FULL TRACEBACK:\n")
                print(tb)

run_spectraPyle_btn = w.Button(
    description="RUN spectraPyle",
    button_style="danger",
    #style=dict(
    #font_weight='bold',
    #text_decoration='underline')
)
        
run_spectraPyle_btn.on_click(run_spectraPyle)

In [33]:
show_log_btn = w.Button(description="Show log")

log_output = w.Output(layout={
    "border": "1px solid black",
    "height": "300px",
    "overflow": "auto"
})

def show_log(_):
    log_output.clear_output()

    path = log_path_w.value
    if not path or not Path(path).exists():
        with log_output:
            print("❌ No log file available")
        return

    with open(path) as f:
        with log_output:
            print(f.read())

show_log_btn.on_click(show_log)

In [34]:
def advanced_box(content, title="⚙️ Advanced"):
    acc = w.Accordion(children=[w.VBox(content)])
    acc.set_title(0, title)
    acc.selected_index = None  # collapsed by default
    return acc

In [35]:
load_gui_tab = w.VBox([
    advanced_box([
        section("Load Existing GUI"),
        load_GUI_fc,
        load_GUI_btn,
        validation_load_gui,
    ]),
])

preview_gui_tab = w.VBox([
    advanced_box([
        section("Preview"),
        preview_btn, 
        preview_out,
    ]),
])

export_raw_gui_tab = w.VBox([
    advanced_box([
        section("Export GUI"),
        save_GUI_name_w,
        save_GUI_btn,
        export_GUI_output,
    ]),
])

export_config_tab = w.VBox([
    advanced_box([
        section("Export JSON/YAML"),
        save_config_name_w,
        save_JSON_btn,
        export_JSON_output,
        save_YAML_btn,
        export_YAML_output,
    ]),
])


In [36]:
instrument_tab = w.VBox([
    section("Instrument (*)"),
    instrument_w,
    survey_w,
    w.HTML('<b>Grisms</b>'),
    grism_section_box,
    datarelease_w,

    advanced_box([
        instrument_qc_box,
    ])
])

In [37]:
processing_tab = w.VBox([
    section("Processing"),
    section("Redshift"),
    ztype_w,
    z_custom_box,
    
    section("Normalization"),
    norm_w,
    norm_dynamic_box,
    
    section("Resampling"),
    resampling_w,
    resampling_note,
    resampling_dynamic_box,
    instrumental_resolution_note,
    
    section("Refining wavelength range"),
    advanced_box([
        lambda_enable_w,
        lambda_box,
        edges_enable_w,
        edges_box,
    ]),

    section("Performance"),
    advanced_box([
        section("Sigma clipping"),
        sigma_w,

        section("Bootstrap"),
        bootstrap_R,

        section("Parallel"),
        parallel_enabled,
        parallel_cpu_frac,

        section("Plot"),
        plot_enabled,
    ]),

    section("Cosmology"),
    advanced_box([
        section("Cosmology"),
        cosmo_H0,
        cosmo_Om0,
    ]),
])

In [38]:
io_tab = w.VBox([
    section("Input/Output (*)"),
    section("Catalogue"),
    catalogue_label,
    file_fc,
    section("Input spectra format"),
    spectra_format_w,
    #dirIn_w,
    #dirOut_w,
    #fileIn_w,
    #fileExt_w,
    section("Spectra"),
    spectra_dir_label,
    spectra_dir_box,
    spectra_datafile_box,

    advanced_box([
        section("Output directory"),
        override_output_w,
        dirOut_w,
        section("Output filename. Default: AUTO"),
        output_name_mode_w,
        output_name_box,
    ])
])

In [39]:
catalogue_tab = w.VBox([
    section("Catalogue (*)"),
    ID_column_w,
    redshift_box,
    metadata_box,
    
    advanced_box([
        section("Galactic extintion"),
        gal_ext_corr_w,
        gal_ext_box,
        gal_ext_banner,

        section("Custom normalization parameter"),
        custom_norm_box,
    ])
])

In [40]:
validate_config_tab = w.VBox([
    section("Validate Config"),
    validate_btn,
    validation_output,
])

In [41]:
actions_tab = w.VBox([
    run_spectraPyle_btn,
    run_spec_output,
])


In [42]:
show_log_tab = w.VBox([
    show_log_btn,
    log_output,
])

In [43]:
# --------------------------------------------------
# TOP ROW: Config management
# --------------------------------------------------
top_tabs = w.Tab(children=[
    load_gui_tab,
    preview_gui_tab,
    export_raw_gui_tab,
    export_config_tab,
])

top_tabs.set_title(0, "Load Config")
top_tabs.set_title(1, "Preview Config")
top_tabs.set_title(2, "Export Raw Config")
top_tabs.set_title(3, "Export JSON/YAML Config")


# --------------------------------------------------
# MIDDLE TOP ROW: Core configuration
# --------------------------------------------------
main_tabs = w.Tab(children=[
    instrument_tab,
    processing_tab,
    io_tab,
    catalogue_tab,
])

main_tabs.set_title(0, "Instrument (*)")
main_tabs.set_title(1, "Stack Params")
main_tabs.set_title(2, "Input/Output (*)")
main_tabs.set_title(3, "Catalogue (*)")

# --------------------------------------------------
# MIDDLE BOTTOM ROW: Run
# --------------------------------------------------
validate_box = w.VBox([
    validate_config_tab,
])


# --------------------------------------------------
# BOTTOM ROW: Run
# --------------------------------------------------
run_box = w.VBox([
    section("Run spectraPyle"),
    actions_tab
])

# --------------------------------------------------
# BOTTOM -1 ROW: Show last log
# --------------------------------------------------
show_log_box = w.VBox([
    section("Log"),
    show_log_tab
])

# --------------------------------------------------
# FINAL LAYOUT
# --------------------------------------------------
ui = w.VBox([
    top_tabs,
    main_tabs,
    validate_box,
    run_box,
    show_log_box
])

In [44]:
from IPython.display import display, HTML

display(ui)